In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from sentence_transformers import SentenceTransformer
import faiss



/Users/topgjulian/Desktop/Projects/AI/RagSume/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
# Python Script for storing documents 
KB = Path('knowledge_base')
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
documents = []

for item in KB.iterdir():
    if Path(item).is_dir():
        for file in item.iterdir():
            if file.suffix == '.md':
                with open(file, "r", encoding="utf-8") as f:
                    chunks = markdown_splitter.split_text(f.read())
                    for chunk in chunks:
                        chunk.metadata["source"] = str(file)
                        documents.append(chunk)

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc.metadata)



Loaded 36 documents
{'Header 1': 'Relevant Coursework', 'Header 2': 'Machine Learning', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Reinforcement Learning', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Data Science', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Recommender Systems', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Relevant Coursework', 'Header 2': 'Software Engineering', 'source': 'knowledge_base/education/relevant_coursework.md'}
{'Header 1': 'Computer Science Undergrad Degree At Dalhousie', 'Header 2': 'Degree Overview', 'source': 'knowledge_base/education/degree.md'}
{'Header 1': 'Computer Science Undergrad Degree At Dalhousie', 'Header 2': 'Graduation', 'source': 'knowledge_base/education/degree.md'}
{'Header 1': 'Computer Science Undergrad Degree

In [40]:
# Embedding Model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)
chunk_content = [doc.page_content for doc in documents]
embeddings = embedding_model.encode(chunk_content, show_progress_bar=True, convert_to_numpy=True)
print(embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(index.ntotal == embeddings.shape[0])
print(index.ntotal)

Batches: 100%|██████████| 2/2 [00:00<00:00, 11.04it/s]

(36, 384)
True
36
